# 5. Choose Chunk Size

Measure retrieval hit-rate for several chunk sizes instead of guessing.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Chunk size is the single most influential knob in RAG, and the right value depends on your documents and questions. Too small: chunks lack context and answers get cut in half. Too large: each embedding averages many topics so similarity becomes blurry, and you waste prompt tokens. Instead of folklore, **measure** it.

## How it works

1. Build a golden set (question + evidence phrase)
2. For each candidate chunk size, rebuild the index
3. Measure hit rate@k and MRR
4. Plot the trade-off and choose the knee of the curve
5. Confirm with end-to-end answer quality

```text
 sizes [150 ... 2400] --> chunk --> index --> retrieve for each golden question --> hit rate, MRR
                                                                                      |
                                                                    pick best size  <-
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** pandas

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
import pandas as pd

text = load_text()
questions = load_eval_questions()

For every chunk size, rebuild the index and check whether the top-k chunks contain the evidence phrase for each question.

In [ ]:
rows = []
for size in [150, 300, 600, 1200, 2400]:
    store = build_store(chunk_text(text, size=size, overlap=size // 10))
    hits_at_3 = [hit_at_k(store.search(q["question"], 3), q["evidence"]) for q in questions]
    rows.append({"chunk_size": size, "n_chunks": len(store), "hit_rate@3": sum(hits_at_3) / len(questions)})
pd.DataFrame(rows)

Small chunks are precise but miss context; huge chunks dilute the embedding. Pick the size where the curve flattens, then confirm with generation quality.

## Add MRR and overlap to the experiment
Hit rate says *whether* the evidence was found; MRR says *how high* it ranked. Overlap also matters - it prevents an idea being split at a boundary.

In [ ]:
rows = []
for size in [150, 300, 600, 1200, 2400]:
    for overlap_pct in [0, 10, 25]:
        st = build_store(chunk_text(text, size=size, overlap=size * overlap_pct // 100))
        r = evaluate_retriever(st.search, questions, k=3)
        rows.append({"size": size, "overlap%": overlap_pct, "chunks": len(st), "hit@3": r["hit_rate@3"], "MRR": r["MRR"]})
grid = pd.DataFrame(rows)
grid.pivot(index="size", columns="overlap%", values="hit@3")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for pct, g in grid.groupby("overlap%"):
    ax[0].plot(g["size"], g["hit@3"], marker="o", label=f"overlap {pct}%")
    ax[1].plot(g["size"], g["MRR"], marker="o", label=f"overlap {pct}%")
for a, t in zip(ax, ["Hit rate@3", "MRR"]):
    a.set_xscale("log"); a.set_xlabel("chunk size (chars)"); a.set_title(t); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Token cost of big chunks
Bigger chunks may retrieve well but every query then sends k x size characters to the LLM.

In [ ]:
cost = pd.DataFrame([{"size": s, "prompt_chars_for_k3": 3 * s, "approx_tokens": 3 * s // 4} for s in [150, 300, 600, 1200, 2400]])
cost

## Which questions fail at each size?
Per-question ranks show *where* a size hurts (None = evidence not retrieved).

In [ ]:
detail = {}
for size in [150, 600, 2400]:
    st = build_store(chunk_text(text, size=size, overlap=size // 10))
    detail[size] = evaluate_retriever(st.search, questions, k=3)["ranks"]
pd.DataFrame(detail, index=[q["question"][:50] for q in questions])

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Turns an argument into a measurement | Needs a golden set (even 20-50 questions helps) |
| Cheap: no LLM calls needed for retrieval-only metrics | Evidence-phrase matching can miss paraphrased evidence |
| Reusable for any corpus | Optimum may differ per document type |

## When to use it

Before any other tuning. Re-run whenever the corpus type or embedding model changes.

## Experiments to try

- Replace the character chunker with a sentence-based one and repeat.
- Try `all-mpnet-base-v2` embeddings; does the best size shift?
- Add your own 10 questions to `data/eval_questions.json`.

## Key takeaways

- Measure, don't guess.
- Mid-sized chunks (a few hundred characters here) typically win for factual Q&A.
- Larger chunks help summarisation-style questions but cost tokens.

---
**Next:** [6. Proposition Chunking](../06_proposition_chunking/proposition_chunking.ipynb)